# Train YOLOv6-nano on a Drone Dataset (Oak-D / RVC2 deploy)

Fine-tunes YOLOv6-nano on a custom drone dataset and exports a `.blob` that can be loaded by DepthAI v3 on an Oak-D S2 (RVC2 / Myriad X).

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

Pipeline:
1. Install YOLOv6 + deps
2. Pull dataset (Roboflow *or* Drive zip)
3. Train `yolov6n` from COCO pretrained
4. Evaluate + spot-check inference
5. Export `.pt` -> ONNX -> OpenVINO `.blob` (6 SHAVES, OpenVINO 2022.1)
6. Generate the DepthAI YOLO JSON config
7. Save artifacts to Drive

## 0. Paths & knobs (edit here only)

In [ ]:
# === EDIT ME ===
PROJECT_DIR   = '/content/project'
YOLOV6_DIR    = f'{PROJECT_DIR}/YOLOv6'
DATASET_DIR   = f'{PROJECT_DIR}/dataset'        # must end up with data.yaml + images/ + labels/
RUNS_DIR      = f'{PROJECT_DIR}/runs'
EXPORT_DIR    = f'{PROJECT_DIR}/export'

CLASS_NAMES   = ['drone']                        # add more if your dataset has them
NUM_CLASSES   = len(CLASS_NAMES)

IMG_SIZE      = 640
BATCH_SIZE    = 32          # drop to 16 if T4 OOMs
EPOCHS        = 100
PATIENCE      = 20          # early-stopping patience (epochs)

# Export
SHAVES        = 6
OPENVINO_VER  = '2022.1'    # RVC2 toolchain
CONF_THRES    = 0.35        # baked into the DepthAI JSON (can be overridden at runtime)
IOU_THRES     = 0.45

import os
for d in (PROJECT_DIR, DATASET_DIR, RUNS_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)
print('Paths ready.')

## 1. Install YOLOv6

In [ ]:
!nvidia-smi | head -n 20

In [ ]:
%cd {PROJECT_DIR}
![ -d YOLOv6 ] || git clone https://github.com/meituan/YOLOv6.git
%cd {YOLOV6_DIR}
# Pin to a known-good commit so the notebook is reproducible
!git checkout 55d80c3 || true
!pip install -q -r requirements.txt
!pip install -q onnx onnxruntime onnxsim blobconverter

In [ ]:
# Grab COCO-pretrained yolov6n weights as the starting checkpoint
%cd {YOLOV6_DIR}
!mkdir -p weights
!wget -q -O weights/yolov6n.pt https://github.com/meituan/YOLOv6/releases/download/0.4.0/yolov6n.pt
!ls -lh weights/

## 2. Dataset

Pick **one** of the two cells below and run it. Both must result in `DATASET_DIR` containing `data.yaml` pointing at train/val image dirs.

Expected layout after this section:
```
$DATASET_DIR/
  data.yaml
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

### 2a. Option A - Roboflow

In [ ]:
# === EDIT ME ===
ROBOFLOW_API_KEY = ''        # paste key
ROBOFLOW_WORKSPACE = ''
ROBOFLOW_PROJECT = ''
ROBOFLOW_VERSION = 1

if ROBOFLOW_API_KEY:
    !pip install -q roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    ds = project.version(ROBOFLOW_VERSION).download('yolov6', location=DATASET_DIR)
    print('Downloaded to', ds.location)
else:
    print('Skipping Roboflow (no API key). Use Option B.')

### 2b. Option B - Zip from Google Drive

In [ ]:
# === EDIT ME ===
DRIVE_ZIP_PATH = ''   # e.g. '/content/drive/MyDrive/datasets/drones.zip'

if DRIVE_ZIP_PATH:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    !rm -rf {DATASET_DIR} && mkdir -p {DATASET_DIR}
    !unzip -q '{DRIVE_ZIP_PATH}' -d {DATASET_DIR}
    !ls {DATASET_DIR}
else:
    print('Skipping Drive zip (no path).')

### 2c. Write / verify `data.yaml`

If your dataset already ships a `data.yaml`, the cell will just print it. Otherwise it writes one from the paths below.

In [ ]:
import os, yaml, glob

DATA_YAML = f'{DATASET_DIR}/data.yaml'

if not os.path.exists(DATA_YAML):
    train_img = f'{DATASET_DIR}/images/train'
    val_img   = f'{DATASET_DIR}/images/val'
    assert os.path.isdir(train_img) and os.path.isdir(val_img), \
        f'Expected {train_img} and {val_img}. Adjust DATASET_DIR or write data.yaml manually.'
    cfg = {
        'train': train_img,
        'val': val_img,
        'is_coco': False,
        'nc': NUM_CLASSES,
        'names': CLASS_NAMES,
    }
    with open(DATA_YAML, 'w') as f:
        yaml.safe_dump(cfg, f)

print(open(DATA_YAML).read())
print('train imgs:', len(glob.glob(f'{DATASET_DIR}/images/train/*')))
print('val imgs:  ', len(glob.glob(f'{DATASET_DIR}/images/val/*')))

## 3. Train

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR}

In [ ]:
%cd {YOLOV6_DIR}
!python tools/train.py \
    --batch {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --img-size {IMG_SIZE} \
    --conf configs/yolov6n_finetune.py \
    --data {DATA_YAML} \
    --device 0 \
    --output-dir {RUNS_DIR} \
    --name drone_yolov6n \
    --stop_aug_last_n_epoch 15

In [ ]:
BEST_PT = f'{RUNS_DIR}/drone_yolov6n/weights/best_ckpt.pt'
assert os.path.exists(BEST_PT), f'Training did not produce {BEST_PT}'
print(BEST_PT)

## 4. Evaluate + spot-check

In [ ]:
%cd {YOLOV6_DIR}
!python tools/eval.py \
    --data {DATA_YAML} \
    --weights {BEST_PT} \
    --img-size {IMG_SIZE} \
    --device 0

In [ ]:
import glob, random, os
from IPython.display import Image, display

SAMPLE_DIR = f'{RUNS_DIR}/drone_yolov6n/sample_infer'
os.makedirs(SAMPLE_DIR, exist_ok=True)
val_imgs = glob.glob(f'{DATASET_DIR}/images/val/*')
random.shuffle(val_imgs)
picks = val_imgs[:4]

for p in picks:
    !python tools/infer.py --weights {BEST_PT} --source '{p}' --yaml {DATA_YAML} --save-dir {SAMPLE_DIR} --device 0 --img-size {IMG_SIZE}

for p in glob.glob(f'{SAMPLE_DIR}/*')[:4]:
    display(Image(filename=p))

## 5. Export: `.pt` -> ONNX -> `.blob`

DepthAI on RVC2 needs an OpenVINO `.blob`. Target OpenVINO **2022.1** and **6 SHAVES** (matches the Oak-D S2 default).

In [ ]:
# 5a. PT -> ONNX. Leave --end2end OFF so DepthAI's YOLO parser gets raw head outputs.
%cd {YOLOV6_DIR}
ONNX_PATH = f'{EXPORT_DIR}/best.onnx'
!python deploy/ONNX/export_onnx.py \
    --weights {BEST_PT} \
    --img-size {IMG_SIZE} {IMG_SIZE} \
    --batch-size 1 \
    --simplify \
    --device 0

import shutil
src = BEST_PT.replace('.pt', '.onnx')
assert os.path.exists(src), f'ONNX not found at {src}'
shutil.move(src, ONNX_PATH)
print('ONNX:', ONNX_PATH)

In [ ]:
# 5b. Sanity-check the ONNX graph before spending time on blob conversion
import onnxruntime as ort, numpy as np, cv2, glob
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name
inp_shape = sess.get_inputs()[0].shape
print('ONNX input:', inp_name, inp_shape)

sample = glob.glob(f'{DATASET_DIR}/images/val/*')[0]
img = cv2.imread(sample)
img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
x = img_r[:, :, ::-1].transpose(2, 0, 1).astype(np.float32) / 255.0
x = np.expand_dims(x, 0)
outs = sess.run(None, {inp_name: x})
print('ONNX outputs:', [o.shape for o in outs])

In [ ]:
# 5c. ONNX -> .blob via Luxonis blobconverter (hits tools.luxonis.com under the hood)
import blobconverter

BLOB_PATH = blobconverter.from_onnx(
    model=ONNX_PATH,
    data_type='FP16',
    shaves=SHAVES,
    version=OPENVINO_VER,
    use_cache=False,
    output_dir=EXPORT_DIR,
    optimizer_params=[
        '--scale_values=[255,255,255]',
        '--reverse_input_channels',
    ],
)
print('Blob:', BLOB_PATH)

## 6. DepthAI YOLO JSON config

YOLOv6 is **anchor-free**, so `anchors` / `anchor_masks` are empty.

In [ ]:
import json

JSON_PATH = f'{EXPORT_DIR}/best.json'
cfg = {
    'nn_config': {
        'output_format': 'detection',
        'NN_family': 'YOLO',
        'input_size': f'{IMG_SIZE}x{IMG_SIZE}',
        'NN_specific_metadata': {
            'classes': NUM_CLASSES,
            'coordinates': 4,
            'anchors': [],
            'anchor_masks': {},
            'iou_threshold': IOU_THRES,
            'confidence_threshold': CONF_THRES,
        },
    },
    'mappings': {
        'labels': CLASS_NAMES,
    },
}
with open(JSON_PATH, 'w') as f:
    json.dump(cfg, f, indent=2)
print(open(JSON_PATH).read())

## 7. Save artifacts to Drive

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=False)

DRIVE_OUT = '/content/drive/MyDrive/yolov6n_drone'
os.makedirs(DRIVE_OUT, exist_ok=True)

import shutil
for src in [BEST_PT, ONNX_PATH, BLOB_PATH, JSON_PATH]:
    dst = f'{DRIVE_OUT}/{os.path.basename(src)}'
    shutil.copy(src, dst)
    print('->', dst)

files.download(BLOB_PATH)
files.download(JSON_PATH)

## 8. Load locally in `Detect.py`

Drop `best.blob` + `best.json` into the repo, then replace the zoo lookup in `Detect.py`:

```python
# was:
# model_desc = dai.NNModelDescription(MODEL_SLUG, platform="RVC2")
# nn = pipeline.create(dai.node.DetectionNetwork).build(cam, model_desc, fps=FPS_TARGET)

# now:
nn = pipeline.create(dai.node.DetectionNetwork)
nn.setBlobPath('best.blob')
nn.setConfidenceThreshold(CONF_THRESHOLD)
nn.setNumClasses(1)
nn.setCoordinateSize(4)
nn.setIouThreshold(0.45)
nn.setAnchors([])          # YOLOv6 is anchor-free
nn.setAnchorMasks({})
nn.input.setBlocking(False)
cam.preview.link(nn.input)
```

Also set the camera preview size to **640x640** to match `IMG_SIZE`, and set `ACCEPT_CLASSES = {'drone'}` (or `None` for any).